# Benchmark: Inference Time & Peak Memory by Attention Type

Measures forward-pass **latency** and **peak memory** for all attention variants across two sweeps:

| Sweep | Fixed | Varied |
|---|---|---|
| Varying *T* | B=16, F=256 | T ∈ {750, 1500, 3000} |
| Varying *F* | B=16, T=1500 | F ∈ {128, 256, 512} |

Both **LinT** (single modality) and **LinMulT** (two modalities, same input repeated) are benchmarked.

- Speedup `(×)` shown relative to `softmax` — higher is better.
- Memory shown as MB delta during one forward pass (CUDA: peak allocated; MPS: allocation delta).
- `performer` requires `linalg_qr`, unsupported on MPS without CPU fallback; set `PYTORCH_ENABLE_MPS_FALLBACK=1` to enable.
- **Expected runtime**: several minutes, dominated by `softmax` and `mha` at large T.

In [ ]:
import time

import torch

from linmult import HeadConfig, LinMulT, LinMulTConfig, LinT, LinTConfig

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Device: {DEVICE}")

In [ ]:
# ── Config — edit to match your setup ───────────────────────────────────────
B = 16  # batch size
T_BASE = 1500  # baseline sequence length
F_BASE = 256  # baseline input feature dimension
OUTPUT_DIM = 7  # head output dimension

D_MODEL = 256  # transformer internal dimension
N_HEADS = 8  # attention heads  (head_dim = D_MODEL // N_HEADS = 32)
N_LAYERS = 4  # transformer layers per encoder

T_VARIANTS = [750, 1500, 3000]  # sequence-length sweep
F_VARIANTS = [128, 256, 512]  # feature-dim sweep

WARMUP = 3  # warm-up runs (excluded from timing)
RUNS = 10  # timed runs

ATTENTION_TYPES = ["linear", "flash", "performer", "bigbird", "mha", "softmax"]
# ─────────────────────────────────────────────────────────────────────
print(f"B={B}  T_base={T_BASE}  F_base={F_BASE}")
print(f"d_model={D_MODEL}  num_heads={N_HEADS}  cmt_num_layers={N_LAYERS}")

In [ ]:
def _sync():
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    elif DEVICE == "mps":
        torch.mps.synchronize()


def _measure_mem_mb(fn):
    """Activation memory delta for one forward pass."""
    if DEVICE == "cuda":
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
        fn()
        torch.cuda.synchronize()
        return torch.cuda.max_memory_allocated() / 1024**2
    elif DEVICE == "mps":
        torch.mps.empty_cache()
        torch.mps.synchronize()
        before = torch.mps.current_allocated_memory()
        fn()
        torch.mps.synchronize()
        return max(0, torch.mps.current_allocated_memory() - before) / 1024**2
    return float("nan")


def run_benchmark(build_fn):
    """build_fn() -> (model, forward_callable).
    Returns {'mean_ms', 'std_ms', 'mem_mb'} or {'error': message}.
    """
    try:
        model, fwd = build_fn()
    except Exception as exc:
        return {"error": str(exc)}

    model.eval()
    with torch.no_grad():
        try:
            for _ in range(WARMUP):
                fwd()
        except Exception as exc:
            return {"error": str(exc)}

        times = []
        for _ in range(RUNS):
            _sync()
            t0 = time.perf_counter()
            fwd()
            _sync()
            times.append((time.perf_counter() - t0) * 1e3)

        mem_mb = _measure_mem_mb(fwd)

    mean_ms = sum(times) / len(times)
    std_ms = (sum((t - mean_ms) ** 2 for t in times) / len(times)) ** 0.5
    return {"mean_ms": mean_ms, "std_ms": std_ms, "mem_mb": mem_mb}

In [ ]:
_HEAD = [HeadConfig(name="out", type="simple", output_dim=OUTPUT_DIM)]
_BASE_CFG = dict(
    d_model=D_MODEL,
    num_heads=N_HEADS,
    cmt_num_layers=N_LAYERS,
    bigbird_block_size=64,
    bigbird_num_global_tokens=16,
    bigbird_num_random_tokens=10,
    heads=_HEAD,
)


def _mask(T):
    return (torch.arange(T).unsqueeze(0) < T - max(T // 10, 1)).expand(B, -1).to(DEVICE)


def make_lint(attn, T, F):
    x = torch.rand(B, T, F, device=DEVICE)
    m = _mask(T)
    cfg = LinTConfig(input_feature_dim=F, attention_type=attn, **_BASE_CFG)
    model = LinT(cfg).to(DEVICE)
    return model, lambda: model(x, m)


def make_linmult(attn, T, F):
    x = torch.rand(B, T, F, device=DEVICE)
    m = _mask(T)
    cfg = LinMulTConfig(input_feature_dim=[F, F], attention_type=attn, **_BASE_CFG)
    model = LinMulT(cfg).to(DEVICE)
    return model, lambda: model([x, x], [m, m])

In [ ]:
_LW, _CW = 12, 22  # label width, column width


def _cell_time(r, sf_ms):
    if not r or "error" in r:
        return "N/A"
    v, s = r["mean_ms"], r["std_ms"]
    cell = f"{v:.1f}\u00b1{s:.1f}ms"
    if sf_ms is not None:
        cell += f" ({sf_ms / v:.1f}\u00d7)"
    return cell


def _cell_mem(r, sf_mb):
    if not r or "error" in r:
        return "N/A"
    v = r["mem_mb"]
    if v != v:  # nan
        return "N/A"
    cell = f"{v:.1f}MB"
    if sf_mb is not None and sf_mb > 0:
        pct = (sf_mb - v) / sf_mb * 100
        cell += f" ({pct:+.0f}%)"
    return cell


def print_table(title, results, col_labels, mode):
    SEP = "\u2500" * (_LW + _CW * len(col_labels))
    print(f"\n{title}")
    print(SEP)
    print(f"{'Attention':<{_LW}}" + "".join(f"{lbl:>{_CW}}" for lbl in col_labels))
    print(SEP)
    for attn in ATTENTION_TYPES:
        cells = []
        for col in col_labels:
            r = results.get(attn, {}).get(col)
            sf = results.get("softmax", {}).get(col)
            if mode == "time":
                sf_val = sf["mean_ms"] if sf and "error" not in sf and attn != "softmax" else None
                cells.append(_cell_time(r, sf_val))
            else:
                sf_val = sf["mem_mb"] if sf and "error" not in sf and attn != "softmax" else None
                cells.append(_cell_mem(r, sf_val))
        print(f"{attn:<{_LW}}" + "".join(f"{c:>{_CW}}" for c in cells))
    print(SEP)
    if mode == "time":
        print("  (\u00d7) speedup vs softmax \u2014 higher is better")
    else:
        print("  (+%) memory saved vs softmax \u2014 positive = uses less memory")
    print()

## Section 1 — Varying Time Dimension

**Fixed:** B=16, F=256  ·  **Varied:** T ∈ {750, 1500, 3000}

Linear-complexity mechanisms (linear, flash, performer, bigbird) should scale roughly linearly;
quadratic mechanisms (mha, softmax) should scale quadratically.

In [ ]:
print("Running LinT — Varying T ...")
res_lt = {a: {} for a in ATTENTION_TYPES}
for attn in ATTENTION_TYPES:
    for T in T_VARIANTS:
        print(f"  {attn:12s}  T={T}  ...", end="\r", flush=True)
        res_lt[attn][f"T={T}"] = run_benchmark(lambda a=attn, t=T: make_lint(a, t, F_BASE))
print(" " * 50)

_cols_t = [f"T={t}" for t in T_VARIANTS]
print_table(f"LinT  |  Inference Time  [B={B}, F={F_BASE}]", res_lt, _cols_t, "time")
print_table(f"LinT  |  Peak Memory     [B={B}, F={F_BASE}]", res_lt, _cols_t, "mem")

In [ ]:
print("Running LinMulT — Varying T ...")
res_mt = {a: {} for a in ATTENTION_TYPES}
for attn in ATTENTION_TYPES:
    for T in T_VARIANTS:
        print(f"  {attn:12s}  T={T}  ...", end="\r", flush=True)
        res_mt[attn][f"T={T}"] = run_benchmark(lambda a=attn, t=T: make_linmult(a, t, F_BASE))
print(" " * 50)

print_table(f"LinMulT  |  Inference Time  [B={B}, F={F_BASE}, 2 mods]", res_mt, _cols_t, "time")
print_table(f"LinMulT  |  Peak Memory     [B={B}, F={F_BASE}, 2 mods]", res_mt, _cols_t, "mem")

## Section 2 — Varying Feature Dimension

**Fixed:** B=16, T=1500  ·  **Varied:** F ∈ {128, 256, 512}

All attention types share the same d_model, so differences here come from the input
projection (Conv1d) and FFN layers, not the attention mechanism itself.

In [ ]:
print("Running LinT — Varying F ...")
res_lf = {a: {} for a in ATTENTION_TYPES}
for attn in ATTENTION_TYPES:
    for F in F_VARIANTS:
        print(f"  {attn:12s}  F={F}  ...", end="\r", flush=True)
        res_lf[attn][f"F={F}"] = run_benchmark(lambda a=attn, f=F: make_lint(a, T_BASE, f))
print(" " * 50)

_cols_f = [f"F={f}" for f in F_VARIANTS]
print_table(f"LinT  |  Inference Time  [B={B}, T={T_BASE}]", res_lf, _cols_f, "time")
print_table(f"LinT  |  Peak Memory     [B={B}, T={T_BASE}]", res_lf, _cols_f, "mem")

In [ ]:
print("Running LinMulT — Varying F ...")
res_mf = {a: {} for a in ATTENTION_TYPES}
for attn in ATTENTION_TYPES:
    for F in F_VARIANTS:
        print(f"  {attn:12s}  F={F}  ...", end="\r", flush=True)
        res_mf[attn][f"F={F}"] = run_benchmark(lambda a=attn, f=F: make_linmult(a, T_BASE, f))
print(" " * 50)

print_table(f"LinMulT  |  Inference Time  [B={B}, T={T_BASE}, 2 mods]", res_mf, _cols_f, "time")
print_table(f"LinMulT  |  Peak Memory     [B={B}, T={T_BASE}, 2 mods]", res_mf, _cols_f, "mem")